# שיעור 30 — איכות נתונים, הנדסת מאפיינים ואתיקה

**הקשר ללמידת מכונה:** דודו קיבל קובץ הזמנות. אותה הזמנה יובאה פעמיים, שתי הזמנות שונות נראות דומות, ואחת מכילה משך שלילי. מחיקה אוטומטית עלולה למחוק עסקה אמיתית.

**עוגן:** [סיכום מפגש 9](../../10-Data-Preparation/Meeting-09-Didactic-Summary.md). הנתונים כאן מומצאים במוצהר; מבנה המשימות הוא התאמה חדשה לרעיונות כפילויות,ערכים חשודים,תאריכים וקידוד.

המחברת היא התאמה דידקטית חדשה. אין קובץ מקור חדש מהמרצה ולכן אין Exact-Original מלאכותי. אין לייחס לה אישור כיסוי מחוון עדכני.

## למה לומדים את זה עכשיו?
אותה הזמנה הועתקה פעמיים; שתי הזמנות שונות יכולות להיראות זהות.
**בתהליך הפרויקט:** איכות והכנה.
**מסגרת פעילה:** 2 שעות לימוד — 90 דקות, כולל דוגמה ותרגול עצמאי. המורה בוחר את משימות הליבה שבמדריך; שאר הפעילויות לחזרה או לעיון, ללא תוספת שעות.

## מושגים וידע קודם
שומרים raw ללא שינוי,יוצרים עותק עבודה ויומן לפני/אחרי. duplicated אינו פסק דין: כפילות על מזהה עסקי שונה מכפילות מדידות. חוסר,אפס וחריג גרפי הם מצבים שונים. בפרויקט אין לפרסם מזהים אישיים; החלפת שם במספר אינה מבטיחה אנונימיות. בודקים רישוי ומקור איסוף,ייצוג אוכלוסייה וקבוצות שהמודל עלול להחמיץ. ייעוד מידע משנה משמעות של שימוש ראוי.

## הסבר מתמטי
מאפיין מחושב חייב להיות זמין בזמן התחזית. סכום=כמות×מחיר יחידה הוא מאפיין חוקי רק אם שני הקלטים כבר ידועים בעת השימוש. אין להכניס תוצאת משלוח סופית לחיזוי אם יהיה איחור. חוקים מבניים כגון מזהה הזמנה ייחודי נבדקים לפני חלוקה; ספים סטטיסטיים שנלמדים מהנתונים נלמדים באימון בלבד. תאריך מפורק לחודש/יום בשבוע; מספר יום בשבוע אינו בהכרח סולם מרחק מתאים.

## נתוני הזמנות מומצאים וחוזה
order_id ייחודי לכל הזמנה; משך ומשקל אינם שליליים. החוזה נתון לצורך התרגיל, ולא חוק לכל עסק.

In [ ]:
import pandas as pd
import numpy as np
raw = pd.DataFrame({
 "order_id":[101,102,103,103,104,105],
 "ordered_at":["2026-01-02","2026-01-03","2026-01-04","2026-01-04","2026-01-05","2026-01-06"],
 "items":[2,2,1,1,3,2], "unit_price":[10.,10.,20.,20.,15.,10.],
 "duration_min":[12.,14.,8.,8.,-3.,None]
})
work = raw.copy(deep=True)
display(work[work.duplicated(subset=["order_id"], keep=False)])
display(work[work.duplicated(subset=["items","unit_price"], keep=False)])
print("Rows before:",len(work))

## ניקוי מנומק
כפילות 103 זהה בכל שדותיה ולכן מסירים את עותק הייבוא. הזמנות אחרות עם אותם פריטים ומחיר נשארות. משך שלילי מסומן כחסר; אין להשלים אותו אם הוא היעד.

In [ ]:
work = work.drop_duplicates(subset=["order_id"], keep="first").copy()
invalid_duration = work["duration_min"] < 0
work.loc[invalid_duration, "duration_min"] = np.nan
print("Rows after:",len(work))
display(work.isna().sum())
# אם duration_min הוא היעד, רק שורות בעלות יעד ידוע זכאיות לאימון
eligible = work.dropna(subset=["duration_min"]).copy()
print("Known targets:",len(eligible))

## מאפיינים ותיעוד
items ו־unit_price נמדדים בזמן הזמנה; revenue ידוע אז בחוזה הזה. זמן מסירה בפועל אינו ידוע אז. תאריך מכיל אזור זמן במקרה אמיתי; אין להסיק מכאן שהפרדה אקראית מתאימה לחיזוי עתידי.

In [ ]:
work["ordered_at"] = pd.to_datetime(work["ordered_at"], errors="raise")
work["order_month"] = work["ordered_at"].dt.month
work["order_weekday"] = work["ordered_at"].dt.dayofweek
work["order_value"] = work["items"] * work["unit_price"]
display(work)
decision_log = pd.DataFrame({
 "decision":["duplicate imported ID","negative duration","same items and price","new features"],
 "action":["keep first identical record","mark missing, do not impute target","keep distinct orders","use only fields known at order time"]
})
display(decision_log)

## מסקנה ובדיקת הבנה
תעדו מה נכנס,מה חושב,מה התקבל ומה אי אפשר להסיק. כתבו שני משפטים על הקשר לפרויקט.
לא מחליפים את כל האפסים ב־NaN; לא מוחקים כל חריג; כשאותו מזהה מופיע עם מידע סותר צריך בירור ולא keep-first שקט. כאן שורת הייבוא זהה ולכן הכלל מוצדק. למתקשים עובדים עם שלוש שורות בלבד,למצטיינים מוסיפים מקרה סתירה ומנסחים מדיניות בלי למחוק.

**כרטיס יציאה:** הסבירו פעולה אחת ללא קריאת הקוד ומגבלה אחת של הדוגמה. דוגמאות מומצאות אינן ראיה על אוכלוסייה אמיתית.